# Fase 2 · EDA Estadístico Profundo y Selección de las 10 Features (Producto 1)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Darkops56/CapstoneProject/blob/main/eda/eda_phase2.ipynb)

Este notebook presenta **paso a paso** el **Análisis Exploratorio de Datos (EDA)**, las pruebas de hipótesis estadísticas, el control de multicolinealidad (**VIF**) y la selección matemática de las **10 variables ortogonales finales** para el modelo de estimación de `costo_total_ars`.

### Perfiles de Relocalización Corporativa
- **`FAMILIA`**: prioriza entornos tranquilos (`<= 65 dBA`), menor densidad delictiva, cercanía escolar, mayor superficie, cochera y admisión real de mascotas.
- **`SOLTERO_EJECUTIVO`**: prioriza conectividad con subte, amenities modernos (`score_amenities`) y eficiencia de costo total mensual.

### Mapa de Pasos
0. **Configuración del Entorno (Local / Google Colab)**.
1. **Carga del Dataset Enriquecido** y verificación de integridad.
2. **Las 5 Preguntas de Negocio (Q1–Q5)**.
3. **Estadísticas Descriptivas, Asimetría (`Skewness`) y Normalidad (Shapiro-Wilk)**.
4. **Pruebas de Hipótesis ($lpha = 0.05$)**: Ruido, Subte, Mascotas, Macro-Barrios, Clústeres Urbanos y Dolarización ($\chi^2$).
5. **Análisis de Correlación de Spearman**.
6. **Poder Predictivo (Random Forest + Mutual Information)**.
7. **Control de Multicolinealidad (VIF Iterativo) y Selección de las 10 Features Finales**.
8. **Exportación del Producto 1 (`producto1_dataset_curado.csv` y `eda_stats.json`)**.

## 0. Configuración del Entorno (Local y Google Colab)

Detecta automáticamente si se ejecuta en Google Colab o de forma local, instala librerías faltantes si fuera necesario y resuelve las rutas del repositorio.

In [ ]:
import importlib
import json
import os
import subprocess
import sys
from pathlib import Path
from typing import Any

IN_COLAB = "google.colab" in sys.modules

REQUIRED_PACKAGES = {
    "scipy": "scipy",
    "sklearn": "scikit-learn",
    "statsmodels": "statsmodels",
    "seaborn": "seaborn",
    "matplotlib": "matplotlib",
}
missing_pkgs = [pip_name for mod, pip_name in REQUIRED_PACKAGES.items() if importlib.util.find_spec(mod) is None]
if missing_pkgs:
    print(f"Instalando dependencias faltantes: {missing_pkgs}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing_pkgs])

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_selection import mutual_info_regression
from sklearn.preprocessing import StandardScaler
from statsmodels.stats.outliers_influence import variance_inflation_factor

try:
    from IPython.display import display
except ImportError:
    display = print

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 60)

REPO_URL = "https://github.com/Darkops56/CapstoneProject.git"

def resolve_project_root() -> Path:
    for cand in [Path.cwd(), Path.cwd().parent, Path("/content/CapstoneProject")]:
        if (cand / "producto1").exists() or (cand / "Datasets").exists():
            return cand.resolve()
    if IN_COLAB:
        target = Path("/content/CapstoneProject")
        print(f"Clonando repositorio en {target}...")
        res = subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(target)], capture_output=True, text=True)
        if res.returncode == 0 and target.exists():
            return target.resolve()
    return Path.cwd().resolve()

BASE_DIR = resolve_project_root()
if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

PRODUCTO1_DIR = BASE_DIR / "producto1"
EDA_DIR = BASE_DIR / "eda"
FIGURES_DIR = EDA_DIR / "figures"
PRODUCTO1_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

TARGET_COL = "costo_total_ars"
CANDIDATE_FEATURES = [
    "m2_totales", "m2_cubiertos", "ambientes", "dormitorios", "banos",
    "toilettes", "cocheras", "antiguedad_anios", "es_a_estrenar",
    "apto_mascotas", "score_amenities", "score_amenities_soltero",
    "score_amenities_familia", "ruido_db_promedio", "densidad_delitos_500m",
    "densidad_robos_500m", "indice_gravedad_delitos_500m",
    "escuelas_familias_500m", "distancia_escuela_m", "distancia_subte_m",
    "lineas_subte_600m", "comuna",
]
MANDATORY_FEATURES = ["apto_mascotas", "score_amenities"]
DOMAIN_GROUPS = {
    "fisicas": ["m2_totales", "m2_cubiertos", "ambientes", "dormitorios", "banos", "toilettes", "cocheras", "antiguedad_anios", "es_a_estrenar"],
    "ruido": ["ruido_db_promedio"],
    "seguridad": ["indice_gravedad_delitos_500m", "densidad_robos_500m", "densidad_delitos_500m"],
    "educacion": ["escuelas_familias_500m", "distancia_escuela_m"],
    "transporte": ["distancia_subte_m", "lineas_subte_600m"],
}

print(f"Entorno: {'Google Colab' if IN_COLAB else 'Local'} | BASE_DIR: {BASE_DIR}")

## 1. Carga del Dataset Enriquecido y Verificación de Integridad

Cargamos el dataset enriquecido proveniente de la Fase 1 (ETL). Si no se encuentra el CSV intermedio pero está disponible el pipeline ETL en el repositorio, lo ejecutamos automáticamente; en Colab sin repositorio clonado permite subir el CSV manualmente.

In [ ]:
required_cols_check = set(CANDIDATE_FEATURES + [TARGET_COL, "precio_m2_total_ars", "barrio", "moneda_origen"])
candidate_paths = [
    PRODUCTO1_DIR / "dataset_enriquecido_completo.csv",
    PRODUCTO1_DIR / "dataset_curado.csv",
    Path.cwd() / "dataset_enriquecido_completo.csv",
    Path.cwd() / "dataset_curado.csv",
]

df = None
loaded_from = None
for p in candidate_paths:
    if p.exists():
        tmp = pd.read_csv(p)
        if required_cols_check.issubset(tmp.columns):
            df = tmp
            loaded_from = p
            break

if df is None and (BASE_DIR / "etl" / "etl_pipeline.py").exists() and (BASE_DIR / "Datasets" / "Originales").exists():
    print("CSV enriquecido no encontrado; ejecutando pipeline ETL de Fase 1...")
    from etl import etl_pipeline as etl
    df, _ = etl.run_full_etl()
    loaded_from = "etl.run_full_etl()"

if df is None and IN_COLAB:
    from google.colab import files
    print("Sube el archivo 'dataset_enriquecido_completo.csv' generado por el notebook ETL:")
    uploaded = files.upload()
    fname = next(iter(uploaded.keys()))
    df = pd.read_csv(fname)
    loaded_from = fname

assert df is not None, "No se pudo cargar el dataset enriquecido."
print(f"Dataset cargado desde: {loaded_from}")
print(f"Dimensiones: {df.shape[0]} filas x {df.shape[1]} columnas")
print(f"Nulos en target + 22 features candidatas: {int(df[CANDIDATE_FEATURES + [TARGET_COL]].isna().sum().sum())}")
display(df.head(5))

## 2. Las 5 Preguntas de Negocio (Q1–Q5)

Estas preguntas guían todo el análisis estadístico y la selección de variables para el Producto 2.

In [ ]:
BUSINESS_QUESTIONS = [
    {
        "id": "Q1",
        "pregunta": "¿Qué zonas y segmentos de departamentos de CABA ofrecen el mejor equilibrio entre calidad ambiental (<65 dBA, baja gravedad delictiva) y conectividad/servicios según el perfil del empleado (FAMILIA vs. SOLTERO_EJECUTIVO)?",
        "objetivo_analitico": "Segmentación K-Means sobre entorno urbano + amenities diferenciados por perfil.",
    },
    {
        "id": "Q2",
        "pregunta": "¿Cuál es el impacto estadísticamente significativo de las externalidades urbanas (ruido diurno >65 dBA, densidad/gravedad delictiva, distancia al subte y oferta escolar) sobre el precio por m² y el costo total mensual?",
        "objetivo_analitico": "Pruebas de hipótesis (Mann-Whitney U, ANOVA/Kruskal-Wallis, Spearman) y explicabilidad.",
    },
    {
        "id": "Q3",
        "pregunta": "¿Qué proporción de la oferta presenta sobreprecio o subvaluación frente al precio justo teórico (alquiler + expensas) explicado por atributos físicos, amenities y entorno?",
        "objetivo_analitico": "Base de métricas de asimetría/dispersión y target limpio costo_total_ars para regresión.",
    },
    {
        "id": "Q4",
        "pregunta": "¿Cómo afectan las inconsistencias de publicación (etiquetas contradictorias sobre mascotas, expensas omitidas y coordenadas fuera de rango) a la elegibilidad real para relocalización corporativa?",
        "objetivo_analitico": "Auditoría ETL con reglas NLP con negaciones, imputación por m²/barrio con banderas y limpieza espacial.",
    },
    {
        "id": "Q5",
        "pregunta": "¿Qué subconjunto de máximo 10 variables ortogonales (VIF controlado) concentra mayor poder predictivo e información mutua para estimar costo_total_ars sin redundancia?",
        "objetivo_analitico": "Selección matemática combinando VIF iterativo + Random Forest + Mutual Information.",
    },
]
display(pd.DataFrame(BUSINESS_QUESTIONS))

## 3. Estadísticas Descriptivas, Asimetría (`Skewness`) y Normalidad (Shapiro-Wilk)

Evaluamos medidas de tendencia central, dispersión robusta (**mediana e IQR**), asimetría y curtosis. Analizamos además cómo la transformación `log1p` reduce la asimetría extrema del target (`costo_total_ars`), justificando:
1. Uso de pruebas **no paramétricas** en el EDA.
2. Transformación **`log1p` / `expm1`** en el entrenamiento de XGBoost.

In [ ]:
def compute_descriptive_stats(dataframe: pd.DataFrame, cols: list[str]) -> dict[str, dict[str, float]]:
    res: dict[str, dict[str, float]] = {}
    for col in cols:
        s = pd.to_numeric(dataframe[col], errors="coerce").dropna()
        q1, q3 = float(s.quantile(0.25)), float(s.quantile(0.75))
        skew_val = float(stats.skew(s, bias=False)) if len(s) > 2 else 0.0
        kurt_val = float(stats.kurtosis(s, bias=False)) if len(s) > 3 else 0.0
        skew_log = float(stats.skew(np.log1p(s.clip(lower=0)), bias=False)) if len(s) > 2 and (s >= 0).all() else np.nan
        res[col] = {
            "count": int(len(s)), "mean": round(float(s.mean()), 2), "std": round(float(s.std(ddof=1)), 2),
            "min": round(float(s.min()), 2), "q25": round(q1, 2), "median": round(float(s.median()), 2),
            "q75": round(q3, 2), "iqr": round(q3 - q1, 2), "max": round(float(s.max()), 2),
            "skewness": round(skew_val, 3), "kurtosis": round(kurt_val, 3),
            "skewness_log1p": round(skew_log, 3) if not np.isnan(skew_log) else None,
        }
    return res

desc_cols = ["costo_total_ars", "alquiler_ars", "expensas_ars", "precio_m2_total_ars", *CANDIDATE_FEATURES]
descriptive_stats = compute_descriptive_stats(df, desc_cols)
desc_df = pd.DataFrame(descriptive_stats).T
display(desc_df.loc[["costo_total_ars", "alquiler_ars", "expensas_ars", "precio_m2_total_ars", "m2_totales", "ruido_db_promedio", "densidad_delitos_500m", "escuelas_familias_500m", "distancia_subte_m", "score_amenities"]])

# Pruebas de normalidad Shapiro-Wilk
normality_res = {}
for col in ["costo_total_ars", "precio_m2_total_ars", "log_precio_m2"]:
    series = np.log1p(df["precio_m2_total_ars"]) if col == "log_precio_m2" else df[col]
    w_stat, p_val = stats.shapiro(series.sample(min(len(series), 500), random_state=42))
    normality_res[col] = {"W": round(float(w_stat), 4), "p_value": float(p_val), "normal_alpha_05": bool(p_val >= 0.05)}
print("\nPruebas de Normalidad Shapiro-Wilk:")
display(pd.DataFrame(normality_res).T)

# Gráfico 1: Distribución del target original vs log1p
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
sns.histplot(df["costo_total_ars"] / 1e6, bins=35, kde=True, ax=axes[0], color="#1f77b4")
axes[0].set_title("Distribución Costo Total Mensual (Millones ARS)")
axes[0].set_xlabel("Millones ARS")
sns.histplot(np.log1p(df["costo_total_ars"]), bins=35, kde=True, ax=axes[1], color="#2ca02c")
axes[1].set_title("Distribución log1p(Costo Total Mensual)")
axes[1].set_xlabel("log1p(ARS)")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "01_distribucion_target.png", dpi=140)
plt.show()

## 4. Pruebas de Hipótesis Estadísticas ($lpha = 0.05$)

Contrastamos formalmente las hipótesis clave del negocio:
- **H1:** Impacto del **ruido diurno** (`> 65 dBA` vs `<= 65 dBA`) sobre `precio_m2_total_ars`.
- **H2:** Diferencia de `precio_m2_total_ars` según **cercanía al subte** (`<= 600 m` vs `> 600 m`).
- **H3:** Impacto de **`apto_mascotas`** sobre `precio_m2_total_ars`.
- **H4:** Diferencias significativas entre **macro-barrios** ($n \ge 15$).
- **H5:** Separación de valor inmobiliario entre **clústeres urbanos preliminares (`K=3`)**.
- **H6:** Relación entre **barrio y dolarización** ($\chi^2$ de independencia).

In [ ]:
def interpret_p(p: float, alpha: float = 0.05) -> str:
    return "Rechazo H0 (significativa)" if p < alpha else "No rechazo H0 (sin evidencia)"

hypothesis_tests: dict[str, Any] = {}

# H1: Ruido diurno > 65 dBA vs <= 65 dBA
noisy = df.loc[df["ruido_db_promedio"] > 65.0, "precio_m2_total_ars"]
quiet = df.loc[df["ruido_db_promedio"] <= 65.0, "precio_m2_total_ars"]
u_ruido, p_u_ruido = stats.mannwhitneyu(noisy, quiet, alternative="two-sided")
t_ruido, p_t_ruido = stats.ttest_ind(noisy, quiet, equal_var=False)
_, p_lev_ruido = stats.levene(noisy, quiet)
hypothesis_tests["H1_ruido_gt_65_vs_le_65"] = {
    "n_ruidoso": int(len(noisy)), "n_tranquilo": int(len(quiet)),
    "median_ruidoso": round(float(noisy.median()), 2), "median_tranquilo": round(float(quiet.median()), 2),
    "estadistico": f"U={u_ruido:.1f}, t={t_ruido:.2f}", "p_value": float(p_u_ruido), "decision": interpret_p(float(p_u_ruido)),
}

# H2: Subte <= 600m vs > 600m
near_subte = df.loc[df["distancia_subte_m"] <= 600.0, "precio_m2_total_ars"]
far_subte = df.loc[df["distancia_subte_m"] > 600.0, "precio_m2_total_ars"]
u_subte, p_u_subte = stats.mannwhitneyu(near_subte, far_subte, alternative="two-sided")
hypothesis_tests["H2_subte_600m"] = {
    "n_cerca": int(len(near_subte)), "n_lejos": int(len(far_subte)),
    "median_cerca": round(float(near_subte.median()), 2), "median_lejos": round(float(far_subte.median()), 2),
    "estadistico": f"U={u_subte:.1f}", "p_value": float(p_u_subte), "decision": interpret_p(float(p_u_subte)),
}

# H3: Apto mascotas
pets_yes = df.loc[df["apto_mascotas"] == 1, "precio_m2_total_ars"]
pets_no = df.loc[df["apto_mascotas"] == 0, "precio_m2_total_ars"]
u_pets, p_u_pets = stats.mannwhitneyu(pets_yes, pets_no, alternative="two-sided")
hypothesis_tests["H3_apto_mascotas"] = {
    "n_apto": int(len(pets_yes)), "n_no_apto": int(len(pets_no)),
    "median_apto": round(float(pets_yes.median()), 2), "median_no_apto": round(float(pets_no.median()), 2),
    "estadistico": f"U={u_pets:.1f}", "p_value": float(p_u_pets), "decision": interpret_p(float(p_u_pets)),
}

# H4: Macro-barrios (n >= 15)
barrio_counts = df["barrio"].value_counts()
top_barrios = barrio_counts[barrio_counts >= 15].index.tolist()
groups_barrio = [df.loc[df["barrio"] == b, "precio_m2_total_ars"].to_numpy() for b in top_barrios]
f_bar, p_f_bar = stats.f_oneway(*groups_barrio)
h_bar, p_h_bar = stats.kruskal(*groups_barrio)
hypothesis_tests["H4_macro_barrios"] = {
    "grupos": len(top_barrios), "estadistico": f"H={h_bar:.2f}, F={f_bar:.2f}",
    "p_value": float(p_h_bar), "decision": interpret_p(float(p_h_bar)),
}

# H5: Clústeres urbanos preliminares (K=3)
env_cols = ["ruido_db_promedio", "indice_gravedad_delitos_500m", "distancia_subte_m", "escuelas_familias_500m"]
km_prelim = KMeans(n_clusters=3, random_state=42, n_init=20).fit_predict(StandardScaler().fit_transform(df[env_cols]))
groups_cl = [df.loc[km_prelim == k, "precio_m2_total_ars"].to_numpy() for k in range(3)]
f_cl, p_f_cl = stats.f_oneway(*groups_cl)
h_cl, p_h_cl = stats.kruskal(*groups_cl)
hypothesis_tests["H5_clusters_urbanos_k3"] = {
    "grupos": 3, "estadistico": f"H={h_cl:.2f}, F={f_cl:.2f}",
    "p_value": float(p_h_cl), "decision": interpret_p(float(p_h_cl)),
}

# H6: Chi2 Barrio vs Moneda
ct = pd.crosstab(df.loc[df["barrio"].isin(top_barrios), "barrio"], df.loc[df["barrio"].isin(top_barrios), "moneda_origen"])
chi2_val, p_chi2, dof, _ = stats.chi2_contingency(ct)
hypothesis_tests["H6_chi2_barrio_moneda"] = {
    "dof": int(dof), "estadistico": f"chi2={chi2_val:.2f}",
    "p_value": float(p_chi2), "decision": interpret_p(float(p_chi2)),
}

display(pd.DataFrame(hypothesis_tests).T[["estadistico", "p_value", "decision"]])

# Gráfico 2: Barrios y Ruido
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
order_barrios = df.groupby("barrio")["precio_m2_total_ars"].median().sort_values(ascending=False).index
sns.boxplot(data=df, x="precio_m2_total_ars", y="barrio", order=order_barrios, ax=axes[0], palette="viridis", hue="barrio", legend=False)
axes[0].set_title("Precio/m² Total (ARS) por Barrio")
axes[0].set_xlabel("ARS / m²")

df_plot = df.copy()
df_plot["zona_ruido"] = np.where(df_plot["ruido_db_promedio"] > 65.0, "Ruidosa (>65 dBA)", "Tranquila (<=65 dBA)")
sns.boxplot(data=df_plot, x="zona_ruido", y="precio_m2_total_ars", ax=axes[1], palette="Set2", hue="zona_ruido", legend=False)
axes[1].set_title("Precio/m² según Ruido Diurno (>65 vs <=65 dBA)")
axes[1].set_xlabel("")
axes[1].set_ylabel("ARS / m²")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "02_barrios_y_ruido.png", dpi=140)
plt.show()

## 5. Análisis de Correlación de Spearman

Calculamos la correlación monótona de Spearman ($\rho$) entre las **22 variables candidatas** y tanto `costo_total_ars` como `precio_m2_total_ars`.

In [ ]:
spearman_rows = []
for col in CANDIDATE_FEATURES:
    r_tot, p_tot = stats.spearmanr(df[col], df["costo_total_ars"])
    r_m2, p_m2 = stats.spearmanr(df[col], df["precio_m2_total_ars"])
    spearman_rows.append({
        "feature": col,
        "rho_costo_total": round(float(r_tot), 4),
        "p_costo_total": float(p_tot),
        "rho_precio_m2": round(float(r_m2), 4),
        "p_precio_m2": float(p_m2),
    })

spearman_df = pd.DataFrame(spearman_rows).sort_values("rho_costo_total", key=lambda s: s.abs(), ascending=False).reset_index(drop=True)
display(spearman_df.head(12))

fig, ax = plt.subplots(figsize=(9, 6))
sp_plot = spearman_df.sort_values("rho_costo_total")
colors_sp = ["#2ca02c" if v >= 0 else "#d62728" for v in sp_plot["rho_costo_total"]]
ax.barh(sp_plot["feature"], sp_plot["rho_costo_total"], color=colors_sp)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_title("Correlación de Spearman (ρ) con costo_total_ars")
ax.set_xlabel("Spearman ρ")
fig.tight_layout()
plt.show()

## 6. Poder Predictivo: Random Forest + Mutual Information

Para medir relaciones tanto lineales como no lineales, entrenamos:
- `RandomForestRegressor` sobre `costo_total_ars` (30%) y sobre `log1p(costo_total_ars)` (30%).
- `mutual_info_regression` sobre `costo_total_ars` (20%) y sobre `log1p(costo_total_ars)` (20%).

Combinamos ambas señales normalizadas en `combined_signal_score ∈ [0, 1]`.

In [ ]:
X = df[CANDIDATE_FEATURES].astype(float)
y = df[TARGET_COL].astype(float)
y_log = np.log1p(y)

rf = RandomForestRegressor(n_estimators=400, max_depth=12, min_samples_leaf=3, random_state=42, n_jobs=-1).fit(X, y)
rf_log = RandomForestRegressor(n_estimators=400, max_depth=12, min_samples_leaf=3, random_state=42, n_jobs=-1).fit(X, y_log)
mi_raw = mutual_info_regression(X, y, random_state=42)
mi_log = mutual_info_regression(X, y_log, random_state=42)

imp_df = pd.DataFrame({
    "feature": CANDIDATE_FEATURES,
    "rf_importance": rf.feature_importances_,
    "rf_log_importance": rf_log.feature_importances_,
    "mutual_info": mi_raw,
    "mutual_info_log": mi_log,
})

for c in ["rf_importance", "rf_log_importance", "mutual_info", "mutual_info_log"]:
    max_v = imp_df[c].max()
    imp_df[f"{c}_norm"] = imp_df[c] / max_v if max_v > 0 else 0.0

imp_df["combined_signal_score"] = (
    0.30 * imp_df["rf_importance_norm"]
    + 0.30 * imp_df["rf_log_importance_norm"]
    + 0.20 * imp_df["mutual_info_norm"]
    + 0.20 * imp_df["mutual_info_log_norm"]
)
imp_df = imp_df.sort_values("combined_signal_score", ascending=False).reset_index(drop=True)
display(imp_df[["feature", "rf_importance", "rf_log_importance", "mutual_info", "mutual_info_log", "combined_signal_score"]].round(4))

## 7. Control de Multicolinealidad (VIF Iterativo) y Selección de las 10 Features

Aplicamos un procedimiento riguroso de selección:
1. Calculamos el **VIF (Variance Inflation Factor)** inicial sobre las 22 candidatas.
2. Eliminamos iterativamente las variables con `VIF >= 5.0` descartando en cada paso la de menor señal predictiva (protegiendo los pilares obligatorios `apto_mascotas` y `score_amenities`).
3. Garantizamos cobertura de todos los dominios del negocio (**físicas**, **`apto_mascotas`**, **`score_amenities`**, **ruido**, **seguridad**, **educación** y **transporte**) y completamos exactamente **10 features ortogonales**.

In [ ]:
def compute_vif_table(dataframe: pd.DataFrame, cols: list[str]) -> pd.Series:
    X_sub = dataframe[cols].astype(float)
    X_std = (X_sub - X_sub.mean()) / X_sub.std(ddof=0).replace(0, 1.0)
    X_const = np.column_stack([np.ones(len(X_std)), X_std.to_numpy()])
    vifs = {col: float(variance_inflation_factor(X_const, i + 1)) for i, col in enumerate(cols)}
    return pd.Series(vifs, name="VIF").sort_values(ascending=False)

initial_vifs = compute_vif_table(df, CANDIDATE_FEATURES)
print("Top 10 VIF iniciales (antes de depurar colinealidad):")
display(initial_vifs.head(10).to_frame())

signal_map = dict(zip(imp_df["feature"], imp_df["combined_signal_score"]))
remaining = list(CANDIDATE_FEATURES)
elimination_log = []
vif_threshold = 5.0

while True:
    vifs = compute_vif_table(df, remaining)
    max_feat, max_vif = vifs.index[0], float(vifs.iloc[0])
    if max_vif < vif_threshold or len(remaining) <= 10:
        break
    high_vif = vifs[vifs >= vif_threshold].index.tolist()
    candidates_to_drop = [f for f in high_vif[:4] if f not in MANDATORY_FEATURES] or [max_feat]
    drop_feat = min(candidates_to_drop, key=lambda f: signal_map.get(f, 0.0))
    remaining.remove(drop_feat)
    elimination_log.append({
        "eliminada": drop_feat,
        "VIF_al_eliminar": round(float(vifs[drop_feat]), 2),
        "signal_score": round(float(signal_map.get(drop_feat, 0.0)), 4),
    })

print("Bitácora de eliminación iterativa por VIF:")
display(pd.DataFrame(elimination_log))

selected_features: list[str] = []
for f in MANDATORY_FEATURES:
    if f in remaining and f not in selected_features:
        selected_features.append(f)

for _, group_cols in DOMAIN_GROUPS.items():
    avail = [c for c in group_cols if c in remaining and c not in selected_features]
    if avail:
        selected_features.append(max(avail, key=lambda c: signal_map.get(c, 0.0)))

rest = sorted([c for c in remaining if c not in selected_features], key=lambda c: signal_map.get(c, 0.0), reverse=True)
for c in rest:
    if len(selected_features) >= 10:
        break
    selected_features.append(c)

selected_features = selected_features[:10]
final_vifs = compute_vif_table(df, selected_features)

summary_top10 = pd.DataFrame({
    "feature": selected_features,
    "combined_signal_score": [round(signal_map[f], 4) for f in selected_features],
    "final_VIF": [round(float(final_vifs[f]), 4) for f in selected_features],
}).sort_values("combined_signal_score", ascending=False).reset_index(drop=True)

print("Las 10 Features Finales Seleccionadas (todas con VIF < 2.5):")
display(summary_top10)

# Gráfico 3: Ranking Feature Importance destacando Top 10
top_imp = imp_df.head(15).sort_values("combined_signal_score", ascending=True)
fig, ax = plt.subplots(figsize=(9, 6))
colors = ["#d62728" if f in selected_features else "#7f7f7f" for f in top_imp["feature"]]
ax.barh(top_imp["feature"], top_imp["combined_signal_score"], color=colors)
ax.set_title("Ranking Combinado de Señal Predictiva (RF + Mutual Info)\n(Rojo = Top 10 Seleccionadas sin Colinealidad)")
ax.set_xlabel("Score Combinado Normalizado")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "03_feature_importance.png", dpi=140)
plt.show()

# Gráfico 4: Heatmap de correlación de las 10 features + target
corr_cols = selected_features + [TARGET_COL]
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(df[corr_cols].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax)
ax.set_title("Matriz de Correlación: 10 Features Finales + Target")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "04_correlacion_top10.png", dpi=140)
plt.show()

## 8. Exportación del Producto 1 (`producto1_dataset_curado.csv`) y Conclusiones

Exportamos el dataset analítico estrictamente acotado a las **10 features seleccionadas + `costo_total_ars`** y el reporte JSON de estadísticas (`eda_stats.json`), listos para alimentar la **Fase 3 (Modelado K-Means, XGBoost y SHAP)**.

In [ ]:
curated_cols = selected_features + [TARGET_COL]
df_curated = df[curated_cols].copy()
output_curated_csv = PRODUCTO1_DIR / "producto1_dataset_curado.csv"
df_curated.to_csv(output_curated_csv, index=False, encoding="utf-8")

eda_report = {
    "business_questions": BUSINESS_QUESTIONS,
    "descriptive_stats": descriptive_stats,
    "normality_shapiro": normality_res,
    "hypothesis_tests": hypothesis_tests,
    "feature_selection": {
        "selected_features": selected_features,
        "target": TARGET_COL,
        "final_vif": {k: round(float(v), 4) for k, v in final_vifs.items()},
        "vif_elimination_log": elimination_log,
        "ranking_table": imp_df.round(5).to_dict(orient="records"),
    },
}
output_eda_json = PRODUCTO1_DIR / "eda_stats.json"
with open(output_eda_json, "w", encoding="utf-8") as f:
    json.dump(eda_report, f, ensure_ascii=False, indent=2)

print(f"Producto 1 exportado: {output_curated_csv}")
print(f"Reporte EDA JSON    : {output_eda_json}")
print(f"Dimensiones: {df_curated.shape[0]} filas x {df_curated.shape[1]} columnas (10 features + target)")
print(f"Nulos totales: {int(df_curated.isna().sum().sum())}")
display(df_curated.head(5))